# OneVoice V2 — fine-tune EnViT5 VI→EN
Checkpoint, optimizer state, lịch sử train và model tốt nhất đều nằm trên Google Drive. Khi Colab ngắt hoặc đổi GPU/tài khoản, mount lại cùng Drive rồi chạy lại từ cell đầu: training tự tiếp tục ở epoch hoàn chỉnh gần nhất.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
DRIVE_ROOT = Path('/content/drive/MyDrive/OneVoice')
CACHE_ROOT = DRIVE_ROOT / 'model_cache'
CHECKPOINT_ROOT = DRIVE_ROOT / 'models/envit5_finetuned_vi2en_v2'
for path in (CACHE_ROOT, CHECKPOINT_ROOT):
    path.mkdir(parents=True, exist_ok=True)
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.environ['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE_ROOT / 'huggingface/hub')
os.environ['TORCH_HOME'] = str(CACHE_ROOT / 'torch')
os.environ['PYTHONUNBUFFERED'] = '1'
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'numpy', 'PyYAML', 'torch', 'transformers==4.57.1', 'tokenizers==0.22.1', 'sentencepiece==0.2.0', 'sacremoses', 'tqdm'], check=True)
# This workflow is text-only; avoid a mismatched optional torchvision binary breaking T5 imports on Colab.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'], check=False)
print('Persistent checkpoint:', CHECKPOINT_ROOT)


In [ ]:
TOTAL_EPOCHS = 3  # Đổi thành 5 để resume từ epoch đã lưu và train tiếp đến epoch 5.
BATCH_SIZE = 8
LEARNING_RATE = 3e-5
TRAIN = REPO / 'data/onevoice_construction_v2/train.csv'
DEV = REPO / 'data/onevoice_construction_v2/dev.csv'
STATE = CHECKPOINT_ROOT / 'training_state.pt'
if STATE.is_file():
    print('Resume is available:', STATE)
else:
    print('No completed epoch yet; this is a new VI→EN run.')
command = [sys.executable, 'scripts/finetune_envit5.py', '--model', 'VietAI/envit5-translation', '--direction', 'vi2en', '--train', str(TRAIN), '--dev', str(DEV), '--output', str(CHECKPOINT_ROOT), '--epochs', str(TOTAL_EPOCHS), '--batch-size', str(BATCH_SIZE), '--learning-rate', str(LEARNING_RATE), '--resume', 'auto']
print('>', ' '.join(command), flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
code = process.wait()
if code:
    raise RuntimeError(f'Fine-tune stopped with exit code {code}. Re-run this cell: it resumes from the latest completed epoch on Drive.')


In [ ]:
import json
history = CHECKPOINT_ROOT / 'training_history.json'
manifest = CHECKPOINT_ROOT / 'run_manifest.json'
if history.is_file():
    display(json.loads(history.read_text(encoding='utf-8')))
if manifest.is_file():
    display(json.loads(manifest.read_text(encoding='utf-8')))
print('Deploy/evaluate checkpoint only:', CHECKPOINT_ROOT / 'best')
